# Inspección de catálogos

Notebook utilitario para cargar **cualquier** catálogo generado por el pipeline
(`.hdf5` o `.pkl`) como un `pandas.DataFrame` y revisarlo visualmente — sin
necesidad de recordar la estructura interna de cada archivo.

Funciona con:
- `catalogo_icl.hdf5` (salida de `00_catalogo.ipynb`)
- `mary_haloshift_z0.pkl` / `mary_subhaloshift_z0.pkl` (catálogos de shift)
- Cualquier otro `.hdf5`/`.pkl` con datasets/campos 1D o 2D (arrays Nx2, Nx3, Nx6, etc.)

Los campos 2D (ej. posiciones `[N,3]`, `SubhaloMassType` `[N,6]`) se expanden
automáticamente en columnas `campo_0`, `campo_1`, ... para que la tabla sea
completamente plana, al estilo pandas.

In [ ]:
import sys, pickle
import numpy as np
import pandas as pd
import h5py
from pathlib import Path
from IPython.display import display

sys.path.insert(0, './original_shift_code')
import params_icl as P

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.4g}')

## Funciones de carga genéricas

In [ ]:
def _expand_2d(name, arr):
    """Convierte un array (N,) o (N,k) en un dict de columnas planas."""
    arr = np.asarray(arr)
    if arr.ndim == 1:
        return {name: arr}
    if arr.ndim == 2:
        return {f'{name}_{j}': arr[:, j] for j in range(arr.shape[1])}
    # ndim >= 3: no es tabular, se ignora (se reporta aparte)
    return {}


def hdf5_to_dataframe(path, group=None, verbose=True):
    """
    Lee un archivo HDF5 (o un grupo interno) y devuelve un DataFrame plano.
    Los datasets escalares (sin filas) se reportan como atributos, no como columnas.
    """
    cols = {}
    scalars = {}
    skipped = []
    with h5py.File(path, 'r') as f:
        node = f[group] if group else f
        attrs = dict(node.attrs)
        n_rows = None
        for key in node.keys():
            ds = node[key]
            if not isinstance(ds, h5py.Dataset):
                continue
            arr = ds[()]
            if np.ndim(arr) == 0:
                scalars[key] = arr
                continue
            if np.ndim(arr) > 2:
                skipped.append(key)
                continue
            cols.update(_expand_2d(key, arr))
            n_rows = len(arr) if n_rows is None else n_rows
    df = pd.DataFrame(cols)
    if verbose:
        print(f"Archivo: {path}")
        if attrs:
            print(f"  Atributos: {attrs}")
        if scalars:
            print(f"  Escalares (no tabulares): {scalars}")
        if skipped:
            print(f"  Datasets omitidos (ndim>2): {skipped}")
        print(f"  Forma de la tabla: {df.shape}")
    return df


def pkl_to_dataframe(path, verbose=True):
    """
    Lee un catálogo pickle (dict de arrays, como mary_haloshift_z0.pkl) y
    devuelve un DataFrame plano. La clave 'count' se reporta aparte.
    """
    with open(path, 'rb') as f:
        raw = pickle.load(f)

    if not isinstance(raw, dict):
        # pkl es directamente un array/lista
        return pd.DataFrame({'value': np.asarray(raw)})

    cols = {}
    scalars = {}
    skipped = []
    for key, val in raw.items():
        arr = np.asarray(val)
        if arr.ndim == 0:
            scalars[key] = arr.item()
            continue
        if arr.ndim > 2:
            skipped.append(key)
            continue
        cols.update(_expand_2d(key, arr))

    df = pd.DataFrame(cols)
    if verbose:
        print(f"Archivo: {path}")
        if scalars:
            print(f"  Escalares: {scalars}")
        if skipped:
            print(f"  Campos omitidos (ndim>2): {skipped}")
        print(f"  Forma de la tabla: {df.shape}")
    return df


def load_catalog(path, group=None, verbose=True):
    """Dispatcher: detecta la extensión y llama a la función adecuada."""
    path = str(path)
    if path.endswith(('.hdf5', '.h5')):
        return hdf5_to_dataframe(path, group=group, verbose=verbose)
    if path.endswith(('.pkl', '.pickle')):
        return pkl_to_dataframe(path, verbose=verbose)
    raise ValueError(f"Extensión no soportada: {path}")

## 1 · Catálogo ICL final (`catalogo_icl.hdf5`)

Salida de `00_catalogo.ipynb` — un cúmulo por fila.

In [ ]:
df_icl = load_catalog(P.CATALOG_OUT)
display(df_icl.head(10).style.background_gradient(cmap='Blues', axis=0))

In [ ]:
df_icl.describe().T

## 2 · Catálogo de halos (`mary_haloshift_z0.pkl`)

Un halo por fila (incluye todos los halos candidatos, fossil + no-fossil + aislados).

In [ ]:
df_halos = load_catalog(P.CATALOG_PKL)
display(df_halos.head(10))

In [ ]:
df_halos[['Group_M_Crit200', 'Group_R_Crit200', 'GroupNsubs']].describe().T

## 3 · Catálogo de subhalos (`mary_subhaloshift_z0.pkl`)

Un subhalo por fila (5000+ filas típicamente — solo se muestran las primeras).

In [ ]:
df_subhalos = load_catalog(P.CATALOG_SUBHALOSHIFT_PKL)
display(df_subhalos.head(10))

In [ ]:
# Masa estelar en log10(M_sun), columna derivada para inspección rápida
if 'SubhaloMassType_4' in df_subhalos:
    df_subhalos['logMstar'] = np.log10(df_subhalos['SubhaloMassType_4'] / P.h + 1e-30) + 10
    display(df_subhalos[['SubhaloGrNr', 'logMstar']].sort_values('logMstar', ascending=False).head(10))

## 4 · Inspección rápida (histogramas)

Usa directamente `DataFrame.plot` / `DataFrame.hist` de pandas — sin necesidad
de escribir matplotlib manualmente para una mirada rápida.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 100})

if 'M200c_Msun' in df_icl:
    np.log10(df_icl['M200c_Msun']).plot.hist(bins=20, color='steelblue', edgecolor='white',
                                              title='log10(M200c) — catalogo_icl.hdf5')
    plt.xlabel(r'$\log_{10}(M_{200c}/M_\odot)$')
    plt.show()

## 5 · Cargar cualquier otro archivo

Cambia `mi_path` por la ruta del catálogo que quieras inspeccionar
(funciona para cualquier `.hdf5`/`.pkl` generado por este pipeline,
incluyendo `fossil_cat_TNG50.hdf5` u otros).

In [ ]:
mi_path = P.CATALOG_OUT   # ← cambia esta ruta
df_generico = load_catalog(mi_path)
display(df_generico.head())